[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-getting-started/blob/main/notebooks/02_knowledge_graph.ipynb)

# 02 · Knowledge Graph — find companies & entities

The **[Bigdata.com](https://bigdata.com)** Knowledge Graph tracks 7M+ companies
plus people, places, products and other entities. Every entity has a stable
6-character **entity ID** (a RavenPack ID, e.g. Apple = `D8442A`). You use these
IDs to filter Search and Research precisely — no ambiguity over names or tickers.

This notebook covers the two most common tasks:
1. **Find a company by details** (name, ticker, website, identifier)
2. **Get companies by market identifier** (ISIN / CUSIP / SEDOL / listing)

In [1]:
import os
import requests

# Your API key is read from the BIGDATA_API_KEY environment variable.
# Never hard-code keys in a notebook you plan to share or commit.
API_KEY = os.environ["BIGDATA_API_KEY"]

# Bigdata exposes two hosts:
#   - api.bigdata.com    -> Knowledge Graph & Search Service (deterministic APIs)
#   - agents.bigdata.com -> Research Agent & Workflows (AI agents, streamed)
API_BASE = "https://api.bigdata.com"
AGENTS_BASE = "https://agents.bigdata.com"

# The key must be sent in the X-API-KEY header on every request.
HEADERS = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

## 1. Find a company by details
`POST /v1/knowledge-graph/companies` accepts a partial/complete **name, website,
ticker, ISIN, SEDOL, or CUSIP** in `query`, with optional filters.

In [2]:
def find_companies(query, **filters):
    """Search the Knowledge Graph for companies. Optional filters:
    types=["PUBLIC"|"PRIVATE"], countries=["US",...], sectors=["Technology",...]."""
    payload = {"query": query, **filters}
    r = requests.post(f"{API_BASE}/v1/knowledge-graph/companies", headers=HEADERS, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()["results"]

results = find_companies("Apple", types=["PUBLIC"])
for c in results[:5]:
    print(f"{c['id']}  {c['name']:<30} {c.get('sector','-'):<14} {c.get('country','-')}  ticker={c.get('ticker','-')}")

D8442A  Apple Inc.                     Technology     US  ticker=AAPL
9D3360  Apple Hospitality REIT Inc.    Real Estate    US  ticker=APLE
72BB2B  Apple Finance Ltd.             Financials     IN  ticker=-
3ORTET  Apple International Co. Ltd.   Consumer Services JP  ticker=2788
85ED92  Apple Flavor & Fragrance Group Co. Ltd. Basic Materials CN  ticker=603020


Grab the top match's entity ID — we'll reuse it as a filter in the Search and Research notebooks.

In [3]:
apple_id = results[0]["id"]
print("Apple entity ID:", apple_id)

Apple entity ID: D8442A


### Refine with filters
Filter by company type, country (ISO 3166-1 alpha-2), and sector. The valid
sector list comes from `GET /v1/knowledge-graph/companies/sectors`.

In [4]:
sectors = requests.get(f"{API_BASE}/v1/knowledge-graph/companies/sectors", headers=HEADERS, timeout=30).json()["results"]
print("Valid sectors:", ", ".join(sectors))

banks = find_companies("bank", types=["PUBLIC"], countries=["US"], sectors=["Financials"])
print(f"\nFound {len(banks)} matches for 'bank' (US, Financials). First few:")
for c in banks[:5]:
    print(f"  {c['id']}  {c['name']}")

Valid sectors: Technology, Financials, Consumer Services, Industrials, Health Care, Consumer Goods, Basic Materials, Telecommunications, Energy, Real Estate, Utilities



Found 20 matches for 'bank' (US, Financials). First few:
  990AD0  Bank of America Corp.
  EF5BED  Bank of New York Mellon Corp.
  10C19B  Bank OZK
  206852  Bank of Hawaii Corp.
  4E162E  Bank of Botetourt


## 2. Get companies by market identifier
When you already hold a standard identifier, resolve it directly. These return a
mapping of `{identifier: company}`.

| Identifier | Endpoint | Example |
|---|---|---|
| ISIN | `/v1/knowledge-graph/companies/isin` | `US0378331005` |
| CUSIP | `/v1/knowledge-graph/companies/cusip` | `037833100` |
| SEDOL | `/v1/knowledge-graph/companies/sedol` | `2046251` |
| Listing (`MIC:ticker`) | `/v1/knowledge-graph/companies/listing` | `XNAS:AAPL` |

In [5]:
def get_by_market_id(kind, values):
    """kind in {"isin","cusip","sedol","listing"}; values is a list of identifiers."""
    r = requests.post(f"{API_BASE}/v1/knowledge-graph/companies/{kind}", headers=HEADERS,
                      json={"values": values}, timeout=30)
    r.raise_for_status()
    return r.json()["results"]

by_isin = get_by_market_id("isin", ["US0378331005"])
for ident, c in by_isin.items():
    print(f"{ident} -> {c['id']}  {c['name']}  ({c.get('sector','-')})")

by_listing = get_by_market_id("listing", ["XNAS:AAPL", "XNAS:MSFT"])
print()
for ident, c in by_listing.items():
    print(f"{ident} -> {c['id']}  {c['name']}")

US0378331005 -> D8442A  Apple Inc.  (Technology)



XNAS:AAPL -> D8442A  Apple Inc.
XNAS:MSFT -> 228D42  Microsoft Corp.


### Look up any entity by ID
`POST /v1/knowledge-graph/entities/id` resolves up to 100 entity IDs at once —
companies, people, places, products, concepts.

In [6]:
r = requests.post(f"{API_BASE}/v1/knowledge-graph/entities/id", headers=HEADERS,
                  json={"values": [apple_id]}, timeout=30)
entity = r.json()["results"][apple_id]
print(entity["name"], "—", entity["category"])
print(entity["description"][:200], "...")

Apple Inc. — companies
Apple Inc. (formerly Apple Computer Inc.), incorporated on January 03, 1977, designs, manufactures, and markets mobile communication and media devices, personal computing products, and portable digita ...


**Key takeaway:** entity IDs (e.g. the Apple ID printed above) are the precise
way to target a company across the Search and Research APIs.

**Next:** [`03_search.ipynb`](03_search.ipynb)

_Source: [Bigdata.com](https://bigdata.com) · KG docs: https://docs.bigdata.com/api-reference/companies/find-by-details_